# Camera Discovery — Harvest-First Orchestration to Balanced Validation

This notebook tests the new one-command orchestration path:

```bash
camera-discovery run ... --harvest-first --harvest-media .m3u8 --harvest-input-mode handoff-only
```

The command should run extraction-only harvest first, write harvest artifacts under `runs/harvest-first-hls-balanced/harvest/`, read the generated `harvest_handoff.json`, and then continue through the normal target-aware discovery/validation pipeline under `runs/harvest-first-hls-balanced/run/`.

Harvested records are raw evidence only. Trusted outputs still require the normal run pipeline's target resolution, deterministic scope gates, media validation, and trust policy.


## Setup

This notebook installs repository code from the configured Git branch and runs the public CLI. It does not patch source files from the notebook.

Default behavior clones the `dev` branch. Change `REPO_BRANCH` or `REPO_URL` when testing a fork or PR branch.


In [ ]:
# Repository setup for Google Colab / notebook execution.
# Change REPO_BRANCH or REPO_URL if testing a fork/PR branch.
REPO_BRANCH = "dev"
REPO_URL = "https://github.com/dshipley71/camera-discovery.git"
REPO_DIR = "/content/camera-discovery"

from pathlib import Path
repo_dir = Path(REPO_DIR)
if not repo_dir.exists():
    !git clone -b "{REPO_BRANCH}" "{REPO_URL}" "{REPO_DIR}"
else:
    print(f"Repository already exists at {repo_dir}. Keeping existing checkout.")
%cd {REPO_DIR}
%pip install -e .[cloakbrowser] --no-build-isolation


## Credentials

Configure `OLLAMA_API_KEY` in Colab secrets if using Ollama Cloud for target resolution and advisory LLM stages. The notebook does not print secrets.


In [ ]:
# Ollama Cloud / LLM credential setup.
# This avoids printing secrets. Configure OLLAMA_API_KEY in Colab: left sidebar > Secrets.
import os

try:
    from google.colab import userdata  # type: ignore
    OLLAMA_API_KEY = userdata.get('OLLAMA_API_KEY')
except Exception:
    OLLAMA_API_KEY = os.environ.get('OLLAMA_API_KEY')

if OLLAMA_API_KEY:
    os.environ['OLLAMA_API_KEY'] = OLLAMA_API_KEY
    os.environ.setdefault('CAMERA_DISCOVERY_LLM_PROVIDER', 'ollama-cloud')
    os.environ.setdefault('CAMERA_DISCOVERY_LLM_MODEL', 'gemma3:27b-cloud')
    os.environ.setdefault('CAMERA_DISCOVERY_TARGET_INTENT_MODEL', 'gemma3:12b-cloud')
    os.environ.setdefault('CAMERA_DISCOVERY_TARGET_INTENT_FALLBACK_MODEL', 'gemma3:12b-cloud')
    os.environ.setdefault('CAMERA_DISCOVERY_GEOCODER_REFEREE_MODEL', 'gemma3:27b-cloud')
    os.environ.setdefault('CAMERA_DISCOVERY_LOCATION_INFERENCE_MODEL', 'gemma3:27b-cloud')
    print('Loaded OLLAMA_API_KEY from Colab userdata/environment')
else:
    print('OLLAMA_API_KEY not found. LLM-backed stages may fail unless another provider is configured.')

# Keep these visible so output records the provider/model, but never print the key.
print('LLM provider:', os.environ.get('CAMERA_DISCOVERY_LLM_PROVIDER', '(default from config)'))
print('LLM model:', os.environ.get('CAMERA_DISCOVERY_LLM_MODEL', '(default from config)'))


## Smoke tests

Verify the CLI, the new run options, and public imports before running the long workflow.


In [ ]:
# CLI and public import smoke tests.
import subprocess, sys
from pathlib import Path

# Editable installs can create a console script that works before the running
# notebook kernel refreshes site-packages. Keep the checked-out source tree on
# sys.path so in-kernel imports use the same repository as the CLI.
repo_src = Path.cwd() / "src"
if repo_src.exists() and str(repo_src) not in sys.path:
    sys.path.insert(0, str(repo_src))

def run_cmd(cmd, *, env=None, check=True):
    print("\n$", " ".join(str(part) for part in cmd))
    result = subprocess.run([str(part) for part in cmd], env=env, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(map(str, cmd))}")
    return result

run_cmd(['camera-discovery', '--help'])
run_help = run_cmd(['camera-discovery', 'run', '--help']).stdout
run_cmd(['camera-discovery', 'harvest-urls', '--help'])
for option in ['--harvest-first', '--harvest-media', '--harvest-max-source-rows', '--harvest-input-mode']:
    assert option in run_help, f'Missing run option: {option}'

from camera_discovery.evidence import ExtractedEvidence
from camera_discovery.services.discovery_engine import CandidateDiscoveryEngine
from camera_discovery.services.harvest_engine import CameraUrlHarvestEngine
import camera_discovery.cli
print('camera-discovery imports OK; ExtractedEvidence:', ExtractedEvidence)


## Notebook-only helper functions

These helpers intentionally live in the notebook, not in `src/`.


In [ ]:
# Notebook-only inspection helpers. These intentionally live in the notebook, not src/.
from __future__ import annotations

import csv
import json
import os
import shutil
import subprocess
from collections import Counter
from pathlib import Path
from urllib.parse import urlparse


def read_json(path):
    path = Path(path)
    if not path.exists():
        print(f"Missing: {path}")
        return None
    return json.loads(path.read_text(encoding='utf-8'))


def print_json(path, keys=None, limit=6000):
    data = read_json(path)
    if data is None:
        return None
    if keys and isinstance(data, dict):
        data = {key: data.get(key) for key in keys}
    text = json.dumps(data, indent=2, sort_keys=True)
    print(text[:limit])
    if len(text) > limit:
        print(f"... truncated {len(text) - limit} characters")
    return data


def iter_jsonl(path, limit=None):
    path = Path(path)
    if not path.exists():
        return
    with path.open('r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            if limit is not None and idx >= limit:
                break
            line = line.strip()
            if not line:
                continue
            yield json.loads(line)


def count_jsonl(path):
    path = Path(path)
    if not path.exists():
        return 0
    with path.open('r', encoding='utf-8') as f:
        return sum(1 for line in f if line.strip())


def media_counts(path):
    counts = Counter()
    for row in iter_jsonl(path):
        counts[str(row.get('media_type') or row.get('camera_type') or 'unknown')] += 1
    return dict(counts)


def top_hosts(path, url_field='url', limit=15):
    counts = Counter()
    for row in iter_jsonl(path):
        url = row.get(url_field) or row.get('media_url') or row.get('stream_url') or row.get('source_url')
        host = urlparse(str(url or '')).netloc.casefold() or '(missing)'
        counts[host] += 1
    return counts.most_common(limit)


def list_existing(paths):
    for path in paths:
        path = Path(path)
        print(('OK      ' if path.exists() else 'MISSING '), path)


def package_output(output_dir):
    output_dir = Path(output_dir)
    if not output_dir.exists():
        print(f'Missing output directory: {output_dir}')
        return None
    archive = shutil.make_archive(str(output_dir), 'zip', root_dir=output_dir)
    print('Wrote archive:', archive)
    try:
        from google.colab import files  # type: ignore
        files.download(archive)
    except Exception:
        print('Download helper unavailable outside Colab. Archive remains on disk.')
    return archive


## Browser backend behavior

This harvest-first validation notebook defaults browser capture off to keep the validation test bounded and reproducible. The application should still preflight and report browser behavior when capture is enabled; do not interpret disabled browser capture as browser success.


In [ ]:
# Browser/backend configuration for this notebook.
BROWSER_BACKEND = "playwright"  # change to "cloakbrowser" when intentionally testing that backend
os.environ['CAMERA_DISCOVERY_ENABLE_BROWSER_CAPTURE'] = 'false'
os.environ['CAMERA_DISCOVERY_BROWSER_BACKEND'] = BROWSER_BACKEND
print('Browser backend selected:', BROWSER_BACKEND)
print('Browser capture enabled:', os.environ['CAMERA_DISCOVERY_ENABLE_BROWSER_CAPTURE'])


## Run harvest-first orchestration through balanced validation

This cell intentionally uses a visible `!camera-discovery run ...` shell command so progress output streams in Colab.

The single command should:

1. run harvest first;
2. write harvest artifacts under `OUTPUT_ROOT / 'harvest'`;
3. read the generated `harvest_handoff.json`;
4. run the normal discovery pipeline under `OUTPUT_ROOT / 'run'`;
5. validate candidates using the selected profile without trusting raw harvest evidence directly.

Use `RERUN_COMBINED=True` to remove prior outputs and rerun.


In [ ]:
%%time
QUERY = "California traffic cameras"
OUTPUT_ROOT = Path('runs/harvest-first-hls-balanced')
RUN_PROFILE = "balanced"
HTTP_TIMEOUT_SECONDS = 10
HARVEST_MAX_SOURCE_ROWS = 1000
RERUN_COMBINED = False

if RERUN_COMBINED and OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

expected = [
    OUTPUT_ROOT / 'harvest' / 'harvest_handoff.json',
    OUTPUT_ROOT / 'harvest' / 'camera_urls.jsonl',
    OUTPUT_ROOT / 'run' / 'logs' / 'run_summary.json',
    OUTPUT_ROOT / 'run' / 'logs' / 'candidate_discovery_summary.json',
]

if not all(path.exists() for path in expected):
    print('Running harvest-first orchestration. Output will stream below.')
    print(f'Profile: {RUN_PROFILE}; HTTP timeout: {HTTP_TIMEOUT_SECONDS}s; harvest source-row cap: {HARVEST_MAX_SOURCE_ROWS}')
    !camera-discovery run "{QUERY}" \
      --harvest-first \
      --harvest-media .m3u8 \
      --harvest-max-source-rows "{HARVEST_MAX_SOURCE_ROWS}" \
      --harvest-input-mode handoff-only \
      --profile "{RUN_PROFILE}" \
      --output-dir "{OUTPUT_ROOT}" \
      --discovery-mode both \
      --browser-backend "{BROWSER_BACKEND}" \
      --http-timeout "{HTTP_TIMEOUT_SECONDS}" \
      --progress-style plain
else:
    print('Skipping combined run: completion artifacts already exist. Set RERUN_COMBINED=True to rerun.')


## Inspect combined output layout and harvest handoff

The harvest directory should contain extraction-only artifacts. The run directory should contain discovery/validation artifacts. Harvested rows should have `source_policy_checked: true`; the handoff should keep `validated`, `trusted`, and `scope_filtered` false.


In [ ]:
# Inspect harvest-first output layout and handoff schema.
HARVEST_DIR = OUTPUT_ROOT / 'harvest'
RUN_DIR = OUTPUT_ROOT / 'run'

print('Output root:', OUTPUT_ROOT)
print('Harvest dir:', HARVEST_DIR)
print('Run dir:', RUN_DIR)
list_existing([
    HARVEST_DIR / 'harvest_summary.json',
    HARVEST_DIR / 'harvest_handoff.json',
    HARVEST_DIR / 'camera_urls.txt',
    HARVEST_DIR / 'camera_urls.csv',
    HARVEST_DIR / 'camera_urls.jsonl',
    HARVEST_DIR / 'camera_records.jsonl',
    HARVEST_DIR / 'camera_media_assets.jsonl',
    HARVEST_DIR / 'discovered_endpoints.jsonl',
    HARVEST_DIR / 'harvest_camera_inventory.jsonl',
    RUN_DIR / 'logs' / 'run_summary.json',
    RUN_DIR / 'logs' / 'candidate_discovery_summary.json',
    RUN_DIR / 'logs' / 'validation_summary.json',
])

handoff = print_json(HARVEST_DIR / 'harvest_handoff.json', keys=[
    'schema_version', 'evidence_schema_version', 'source_provided_only', 'validated', 'geocoded',
    'scope_filtered', 'trusted', 'llm_reviewed', 'media_filter', 'handoff_default_scope', 'counts', 'warnings'
]) or {}
assert handoff.get('schema_version') == 'harvest-handoff/v2'
assert handoff.get('evidence_schema_version') == 'extracted-evidence/v1'
assert handoff.get('validated') is False
assert handoff.get('trusted') is False
assert handoff.get('scope_filtered') is False

print('\nHarvest media counts:', media_counts(HARVEST_DIR / 'camera_urls.jsonl'))
print('Top harvest URL hosts:', top_hosts(HARVEST_DIR / 'camera_urls.jsonl'))
print('JSONL counts:')
for name in ['camera_urls.jsonl', 'camera_records.jsonl', 'camera_media_assets.jsonl', 'discovered_endpoints.jsonl', 'harvest_camera_inventory.jsonl']:
    print(name, count_jsonl(HARVEST_DIR / name))

print('\nSample camera_urls.jsonl rows and source_policy_checked flags:')
for row in iter_jsonl(HARVEST_DIR / 'camera_urls.jsonl', limit=5):
    print(json.dumps({k: row.get(k) for k in ['url', 'media_type', 'source_provider', 'discovery_method', 'source_policy_checked', 'blocked_reason']}, indent=2))
    assert row.get('source_policy_checked') is True

assert not (HARVEST_DIR / 'camera.geojson').exists()
assert not (HARVEST_DIR / 'untrusted_camera_candidates.geojson').exists()
assert not (HARVEST_DIR / 'review_artifacts.zip').exists()


## Inspect discovery and validation outputs

These artifacts come from the normal run pipeline. Candidate rows seeded from harvest input must remain untrusted unless normal target/scope/validation/trust gates authorize otherwise.


In [ ]:
# Inspect normal run outputs.
print('Run directory:', RUN_DIR)
list_existing([
    RUN_DIR / 'logs' / 'run_summary.json',
    RUN_DIR / 'logs' / 'run_explanation.json',
    RUN_DIR / 'logs' / 'candidate_discovery_summary.json',
    RUN_DIR / 'logs' / 'candidate_priority_summary.json',
    RUN_DIR / 'logs' / 'validation_summary.json',
    RUN_DIR / 'logs' / 'validation_priority_summary.json',
    RUN_DIR / 'camera_candidates_table.csv',
    RUN_DIR / 'untrusted_camera_candidates.geojson',
    RUN_DIR / 'camera.geojson',
    RUN_DIR / 'map.html',
    RUN_DIR / 'review_artifacts.zip',
    RUN_DIR / 'media_validation_dashboard.json',
])

print('\nRun summary:')
print_json(RUN_DIR / 'logs' / 'run_summary.json')
print('\nCandidate discovery summary:')
summary = print_json(RUN_DIR / 'logs' / 'candidate_discovery_summary.json') or {}
print('\nValidation summary:')
validation = print_json(RUN_DIR / 'logs' / 'validation_summary.json') or {}
print('\nValidation attempted/skipped:', validation.get('attempted'), validation.get('skipped'))

harvest_input = summary.get('harvest_input') or {}
print('\nHarvest input summary from run:', json.dumps(harvest_input, indent=2, sort_keys=True)[:3000])
assert harvest_input.get('mode') == 'handoff-only'
assert harvest_input.get('normal_discovery_enabled') is False

csv_path = RUN_DIR / 'camera_candidates_table.csv'
if csv_path.exists():
    with csv_path.open(newline='', encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    print('\nCandidate table rows:', len(rows))
    print('Media counts:', dict(Counter(row.get('camera_type') or row.get('media_type') or '(missing)' for row in rows)))
    print('Scope counts:', dict(Counter(row.get('scope_status') or '(missing)' for row in rows)))
    print('Validation counts:', dict(Counter(row.get('validation_status') or '(missing)' for row in rows)))
    print('First 5 candidate rows:')
    for row in rows[:5]:
        print({k: row.get(k) for k in ['camera_type', 'scope_status', 'validation_status', 'trust_level', 'stream_url', 'latitude', 'longitude'] if k in row})
else:
    print('No candidate table found.')

for geojson_name in ['camera.geojson', 'untrusted_camera_candidates.geojson']:
    path = RUN_DIR / geojson_name
    data = read_json(path)
    if data:
        features = data.get('features', [])
        print(f"{geojson_name}: {len(features)} features")
        print('Feature trust counts:', dict(Counter((feat.get('properties') or {}).get('trust_level') or '(missing)' for feat in features)))
        print('Feature scope counts:', dict(Counter((feat.get('properties') or {}).get('scope_status') or '(missing)' for feat in features)))


## Media validation dashboard, playlists, and optional dorking summary

Playlist/TXT exports are derived convenience artifacts only. They do not promote candidates to trusted output.


In [ ]:
# Inspect media validation dashboard, playlists, and optional Google dorking counts.
dashboard_path = RUN_DIR / 'media_validation_dashboard.json'
if dashboard_path.exists():
    print('media_validation_dashboard.json')
    print_json(dashboard_path, limit=5000)
else:
    print('media_validation_dashboard.json not found at', dashboard_path)

playlist_dir = RUN_DIR / 'playlists'
if playlist_dir.exists():
    print('\nplaylist artifacts:')
    for path in sorted(playlist_dir.glob('*')):
        print('-', path.relative_to(RUN_DIR))
else:
    print('playlists/ not found at', playlist_dir)

dork_path = RUN_DIR / 'logs' / 'google_dorking_summary.json'
if dork_path.exists():
    data = read_json(dork_path) or {}
    print('\ngoogle_dorking:', {k: data.get(k) for k in ['enabled', 'queries_generated', 'results_seen', 'results_after_block_policy', 'promoted_source_leads', 'candidates_extracted']})
else:
    print('google_dorking summary not present')


## Passive intelligence summary

This cell consumes source-code-generated passive intelligence artifacts. It does not patch repository source code.


In [ ]:
# Inspect passive intelligence artifacts if the run emitted them.
passive_path = RUN_DIR / 'logs' / 'passive_intelligence_summary.json'
dashboard = read_json(RUN_DIR / 'media_validation_dashboard.json') or {}
passive = read_json(passive_path) if passive_path.exists() else dashboard.get('passive_intelligence', {})
if passive:
    print('candidate_evidence_bands:', passive.get('candidate_evidence_bands', {}))
    print('protocol_label_counts:', passive.get('protocol_label_counts', {}))
    print('signature_family_counts:', passive.get('signature_family_counts', {}))
    reasons = passive.get('top_evidence_reasons') or passive.get('evidence_reasons') or []
    print('top evidence reasons:', reasons[:10] if isinstance(reasons, list) else reasons)
else:
    print('Passive intelligence artifacts not found yet. Run the pipeline first, then rerun this cell.')


## Package outputs

Run this after the workflow completes if you want a downloadable zip of both harvest and run artifacts.


In [ ]:
# Optional: package and download combined outputs.
package_output(OUTPUT_ROOT)
